In [1]:
import cv2
import numpy as np
import time

# ---- USER SETTINGS ----
marker_size_meters = 0.05   # Real printed side length of the marker's black square, in metres
camera_index = 0
# ------------------------

# Load webcam calibration
calibration = np.load("webcam_calibration.npz")
camera_key = next((k for k in ("camera_matrix", "mtx", "cameraMatrix") if k in calibration), None)
dist_key = next((k for k in ("dist_coeffs", "dist", "distCoeffs") if k in calibration), None)
if camera_key is None or dist_key is None:
    raise KeyError(f"Could not find camera matrix and distortion coefficients. File contains: {calibration.files}")
camera_matrix = calibration[camera_key]
dist_coeffs = calibration[dist_key]

# Detector (works on both new and old OpenCV APIs)
dictionary = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_50)
detector = cv2.aruco.ArucoDetector(dictionary) if hasattr(cv2.aruco, "ArucoDetector") else None

# Marker corners in the marker's own frame (order required by SOLVEPNP_IPPE_SQUARE)
half = marker_size_meters / 2
object_points = np.array([
    [-half,  half, 0],
    [ half,  half, 0],
    [ half, -half, 0],
    [-half, -half, 0],
], dtype=np.float32)

cap = cv2.VideoCapture(camera_index)
if not cap.isOpened():
    raise RuntimeError("Could not open webcam")

last_print = 0.0

try:
    while True:
        ok, frame = cap.read()
        if not ok:
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        if detector is not None:
            corners, ids, _ = detector.detectMarkers(gray)
        else:
            corners, ids, _ = cv2.aruco.detectMarkers(gray, dictionary)

        now = time.monotonic()
        do_print = now - last_print >= 0.1

        if ids is None:
            if do_print:
                print("No marker detected")
        else:
            # Green border + ID label on every detected marker
            cv2.aruco.drawDetectedMarkers(frame, corners, ids)

            for marker_corners, marker_id in zip(corners, ids.flatten()):
                image_points = marker_corners.reshape(4, 2).astype(np.float32)
                found, rvec, tvec = cv2.solvePnP(
                    object_points, image_points, camera_matrix, dist_coeffs,
                    flags=cv2.SOLVEPNP_IPPE_SQUARE,
                )
                if not found:
                    continue

                distance = float(np.linalg.norm(tvec))

                # X (red), Y (green), Z (blue) axes from the marker centre
                cv2.drawFrameAxes(frame, camera_matrix, dist_coeffs,
                                  rvec, tvec, marker_size_meters * 0.5)

                # Centre point (projected origin of the marker frame)
                center, _ = cv2.projectPoints(np.zeros((1, 3), dtype=np.float32),
                                              rvec, tvec, camera_matrix, dist_coeffs)
                cx, cy = np.round(center.reshape(2)).astype(int)
                cv2.circle(frame, (cx, cy), 5, (0, 255, 255), -1)

                # ID + distance text above the first corner
                x, y = np.round(image_points[0]).astype(int)
                cv2.putText(frame, f"ID {marker_id}  {distance:.2f} m",
                            (x, max(y - 10, 20)), cv2.FONT_HERSHEY_SIMPLEX,
                            0.6, (0, 255, 0), 2)

                if do_print:
                    print(f"ID {marker_id}: rvec={rvec.ravel()}, "
                          f"tvec={tvec.ravel()} m, distance={distance:.3f} m")

        if do_print:
            last_print = now

        # Show the frame AFTER drawing on it
        cv2.imshow("Marker pose (press q to quit)", frame)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break
finally:
    cap.release()
    cv2.destroyAllWindows()

No marker detected
No marker detected
No marker detected
No marker detected
No marker detected
No marker detected
No marker detected
No marker detected
No marker detected
No marker detected
No marker detected
No marker detected
No marker detected
No marker detected
No marker detected
No marker detected
No marker detected
No marker detected
No marker detected
No marker detected
No marker detected
No marker detected
ID 0: rvec=[ 2.06655714 -1.87781537  0.12659847], tvec=[-0.11706022  0.00492974  0.39367314] m, distance=0.411 m
ID 3: rvec=[ 2.11523963 -1.9990876   0.14424796], tvec=[-0.12176612 -0.05767424  0.45047834] m, distance=0.470 m
ID 0: rvec=[ 1.90697662 -1.90540992  0.07412322], tvec=[-0.10944263  0.00560055  0.40347272] m, distance=0.418 m
ID 3: rvec=[ 1.98617368e+00 -2.05678597e+00 -8.26699531e-04], tvec=[-0.11913239 -0.05666963  0.46424583] m, distance=0.483 m
ID 0: rvec=[ 1.86048354 -1.92838355  0.0110115 ], tvec=[-0.10802073  0.00591604  0.41449566] m, distance=0.428 m
ID 3:

In [ ]:
aruco_id = 3
marker_size_pixels = 600
output_file = f"aruco_id{aruco_id}.png"

if not 0 <= aruco_id < 50:
    raise ValueError("aruco_id must be between 0 and 49")

dictionary = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_50)
marker = cv2.aruco.generateImageMarker(dictionary, aruco_id, marker_size_pixels)

if not cv2.imwrite(output_file, marker):
    raise OSError(f"Could not save marker to {output_file}")
    marker_size_mm = 50.0  # Set to the marker's actual side length in mm

    cap.release()  # The earlier calibration cell released the camera
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        raise RuntimeError("Could not open the webcam")

    parameters = cv2.aruco.DetectorParameters()
    if hasattr(cv2.aruco, "ArucoDetector"):
        detector = cv2.aruco.ArucoDetector(dictionary, parameters)

    while True:
        ret, frame = cap.read()
        if not ret:
            print("[ERROR] Failed to capture image.")
            break

        if hasattr(cv2.aruco, "ArucoDetector"):
            marker_corners, ids, _ = detector.detectMarkers(frame)
        else:
            marker_corners, ids, _ = cv2.aruco.detectMarkers(
                frame, dictionary, parameters=parameters
            )

        if ids is not None:
            cv2.aruco.drawDetectedMarkers(frame, marker_corners, ids)

            for i, detected_id in enumerate(ids.flatten()):
                if detected_id == aruco_id:
                    rvecs, tvecs, _ = cv2.aruco.estimatePoseSingleMarkers(
                        [marker_corners[i]], marker_size_mm,
                        camera_matrix, dist_coeffs
                    )
                    cv2.drawFrameAxes(
                        frame, camera_matrix, dist_coeffs,
                        rvecs[0], tvecs[0], marker_size_mm * 0.5
                    )
                    x, y, z = tvecs[0].flatten()
                    cv2.putText(
                        frame, f"ID {aruco_id}: x={x:.1f}, y={y:.1f}, z={z:.1f} mm",
                        (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6,
                        (0, 255, 0), 2
                    )

        cv2.imshow("ArUco detection", frame)
        if cv2.waitKey(1) & 0xFF == 27:
            break

    cap.release()
    cv2.destroyAllWindows()
print(f"Saved marker ID {aruco_id} to {output_file}")